# ProSIT 2 — V2 Experimental Setup & Model Benchmarks

**Uses `train.csv` only, through verified V2 engineering outputs.** This notebook **never reads `test.csv`**. It builds a chronological final holdout, saved 5-fold random/grouped splits and 3 expanding temporal folds, compares the prescribed syllabus models, and exports development-only comparison tables and figures. **It does not evaluate the final holdout.**

**Target meanings:** Classification `target` (1=default), regression `Total_Amount` (historically disbursed KES; not verified affordable limit).

**Feature sets:** `F0_context`, `F1_borrowing_primary` (recommended), `F2_outcome_sensitivity` (optional, unverified due-date label availability).

## 0 · Paths and reproducibility
Run from the ProSIT2 repository root. You may change `ROOT` if this notebook lives elsewhere. Reuses V2 manifests without guessing column names. Outputs are versioned under `experiments_v2/` to avoid mixing V1 and V2 experiments.

In [ ]:
from pathlib import Path
import os, json, hashlib, time, warnings, sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
import joblib

SEED=42
ROOT=Path.cwd()   # override to your project root if needed
PROCESSED=ROOT/'data'/'processed_v2'
AUDITS=ROOT/'audits'/'v2'
OUTPUT=ROOT/'experiments_v2'
SPLITS=OUTPUT/'splits'; TABLES=OUTPUT/'tables'; FIGURES=OUTPUT/'figures'; PREDICTIONS=OUTPUT/'predictions'
for p in (SPLITS,TABLES,FIGURES,PREDICTIONS): p.mkdir(parents=True,exist_ok=True)
FEATURE_SET='F1_borrowing_primary'
# fast default for today's submission; set to False for only data/split verification.
RUN_BASELINES=True
# Optional grouped and temporal model comparisons are recommended for the actual assignment.
CV_PROTOCOLS=['stratified','grouped','temporal']
# limit n_jobs to avoid exhausting an 8-GB laptop
N_JOBS=2
assert PROCESSED.exists(),f'V2 outputs not found: {PROCESSED.resolve()}; run V2 engineering first.'
print('Python:',sys.version.split()[0],'scikit-learn:',__import__('sklearn').__version__)
print('Project:',ROOT.resolve())

## 1 · Read V2 products, validate identities and feature contracts
**Fail closed** if audits indicate anomalies; do not train on unverified customer identities. We cannot retroactively prove raw input integrity using only the engineered outputs, so also inspect V2 audit reports.

In [ ]:
def json_read(name):
    path=PROCESSED/name
    if not path.is_file(): raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding='utf-8'))

source_manifest=json_read('engineering_manifest_v2.json')
feature_sets={key:json_read(key+'.json') for key in ('F0_context','F1_borrowing_primary','F2_outcome_sensitivity')}
cls_feats=json_read('classification_features.json'); reg_feats=json_read('regression_features.json')
cat_feats=json_read('categorical_features.json')
assert source_manifest.get('output_schema')=='feature_eng_v2'
assert source_manifest.get('input')=='train.csv only'
assert source_manifest.get('primary_uses_outcome_history') is False
assert source_manifest.get('original_customer_ids_preserved') is True
assert feature_sets['F1_borrowing_primary']==cls_feats==reg_feats
assert FEATURE_SET in feature_sets and feature_sets[FEATURE_SET]
if FEATURE_SET=='F2_outcome_sensitivity':
    raise ValueError('F2 requires an explicit outcomes-availability study; this core notebook intentionally excludes it.')

for filename in ('identity_mismatches.csv','invalid_customer_ids.csv'):
    path=AUDITS/filename
    if not path.exists(): raise FileNotFoundError(f'Missing mandatory audit file: {path}')
    audit=pd.read_csv(path)
    if len(audit): raise AssertionError(f'Unresolved data-integrity issue in {path}: {len(audit)} rows; STOP training.')

cls=pd.read_csv(PROCESSED/'classification_dataset.csv',dtype={'ID':'string','customer_id':'string','tbl_loan_id':'string','country_id':'string','lender_id':'string','loan_type':'string'})
reg=pd.read_csv(PROCESSED/'regression_dataset.csv',dtype={'ID':'string','customer_id':'string','tbl_loan_id':'string','country_id':'string','lender_id':'string','loan_type':'string'})
for label,frame in [('classification',cls),('regression',reg)]:
    required={'row_key','ID','customer_id','disbursement_date','due_date'} | set(feature_sets[FEATURE_SET])
    if label=='classification':required.add('target')
    else:required.add('Total_Amount')
    missing=required-set(frame.columns)
    if missing:raise AssertionError(f'{label} missing columns {sorted(missing)}')
    assert frame['row_key'].is_unique and frame['ID'].is_unique
    ids=frame['customer_id'].astype('string').str.strip()
    assert ids.str.fullmatch(r'[1-9][0-9]{5}').fillna(False).all(),f'Invalid customer_id in {label}'
    frame['customer_id']=ids
    frame['disbursement_date']=pd.to_datetime(frame['disbursement_date'],errors='coerce')
    frame['due_date']=pd.to_datetime(frame['due_date'],errors='coerce')
    assert frame.disbursement_date.notna().all()
    assert not frame[feature_sets[FEATURE_SET]].replace([np.inf,-np.inf],np.nan).isna().all(axis=1).any(), 'Found a fully null feature row'
    for f in feature_sets[FEATURE_SET]:
        if f not in cat_feats:frame[f]=pd.to_numeric(frame[f],errors='coerce')
    if label=='classification':
        assert frame.target.isin([0,1]).all()
    else:
        assert pd.to_numeric(frame.Total_Amount,errors='coerce').gt(0).all()

print('V2 classification:',cls.shape,'default prevalence:',f'{cls.target.mean():.3%}')
print('V2 regression:',reg.shape,'positive amount rows:',len(reg))
print('Selected features:',FEATURE_SET,'count:',len(feature_sets[FEATURE_SET]),'categorical:',set(cat_feats)&set(feature_sets[FEATURE_SET]))
print('Source manifest counts:');display(pd.DataFrame([source_manifest['counts']]).T.rename(columns={0:'value'}))

## 2 · Chronological final holdout (never used in model selection)
A shared date cutoff ensures both tasks are evaluated on future loans. Do not move it in response to a bad score. We save row keys and cutoff in a manifest. The holdout is sealed; this notebook **does not calculate holdout metrics**.

In [ ]:
# Compute cutoff on unique calendar dates from all eligible loan snapshots (no target values).
all_dates=pd.concat([cls.disbursement_date,reg.disbursement_date]).dt.normalize().dropna().drop_duplicates().sort_values().reset_index(drop=True)
assert len(all_dates)>=10,'Not enough distinct dates to form reliable chronological splits.'
cutoff=all_dates.iloc[min(len(all_dates)-1,max(1,int(0.8*len(all_dates))))]

def create_holdout(frame,name):
    dates=frame.disbursement_date.dt.normalize()
    dev=frame.loc[dates<cutoff].copy()
    hold=frame.loc[dates>=cutoff].copy()
    assert len(dev)>0 and len(hold)>0
    manifest=pd.DataFrame({'row_key':frame.row_key,'ID':frame.ID.astype(str),'customer_id':frame.customer_id.astype(str),'disbursement_date':dates,'partition':np.where(dates<cutoff,'development','final_holdout')})
    manifest.to_csv(SPLITS/(name+'_holdout_assignments.csv'),index=False)
    print(name,'development',len(dev),'holdout',len(hold),'cutoff',cutoff.date())
    return dev,hold

cls_dev,cls_hold=create_holdout(cls,'classification')
reg_dev,reg_hold=create_holdout(reg,'regression')
if cls_dev.target.nunique()!=2:raise AssertionError('Development classification must contain both labels.')
if cls_hold.target.nunique()<2:warnings.warn('Holdout contains one class: PR-AUC/ROC-AUC will be limited.')
# freeze target-independent cutoff and row assignments
(SPLITS/'holdout_cutoff.json').write_text(json.dumps({'date':str(cutoff.date()),'policy':'loans on/after cutoff are final holdout; do not tune','seed':SEED},indent=2))

split_report=pd.DataFrame([
    {'task':'classification','part':'development','rows':len(cls_dev),'customers':cls_dev.customer_id.nunique(),'defaults':int(cls_dev.target.sum()),'default_rate':float(cls_dev.target.mean())},
    {'task':'classification','part':'holdout','rows':len(cls_hold),'customers':cls_hold.customer_id.nunique(),'defaults':int(cls_hold.target.sum()),'default_rate':float(cls_hold.target.mean())},
    {'task':'regression','part':'development','rows':len(reg_dev),'customers':reg_dev.customer_id.nunique(),'mean_amount':float(reg_dev.Total_Amount.mean()),'median_amount':float(reg_dev.Total_Amount.median())},
    {'task':'regression','part':'holdout','rows':len(reg_hold),'customers':reg_hold.customer_id.nunique(),'mean_amount':float(reg_hold.Total_Amount.mean()),'median_amount':float(reg_hold.Total_Amount.median())},
]);split_report.to_csv(TABLES/'development_holdout_statistics.csv',index=False);display(split_report)

## 3 · Save CV fold assignments
- **StratifiedKFold (5 folds)**: classification benchmark, may have same customer in both partitions.
- **StratifiedGroupKFold (5 folds)** for classification; **GroupKFold (5 folds)** for regression: unseen-customer check.
- **Expanding-window temporal CV (3 folds)**: older-disbursement training followed by newer-loan validation; training rows are conservatively restricted to those with `due_date` strictly before validation starts to approximate label maturity. Since actual resolution timestamps are unavailable, this remains a limitation.
- For temporal CV, the cutoff logic is applied to the **training rows**, not to the history already encoded in V2 snapshots. This is an important provenance distinction.

In [ ]:
def build_temporal_folds(frame,n_splits=3):
    dates=frame.disbursement_date.dt.normalize()
    unique=np.array(sorted(dates.dropna().unique()),dtype='datetime64[ns]')
    # validation windows in latter 60% of unique dates; expanding training history
    cuts=np.linspace(int(len(unique)*0.4),len(unique),n_splits+1,dtype=int)
    out=[]
    for k in range(n_splits):
        start=unique[cuts[k]];end=unique[cuts[k+1]-1]
        val=(dates>=start)&(dates<=end)
        prior=(dates<start)
        # Label maturity: do not fit on observations whose outcomes may not be recorded.
        maturity=frame.due_date.notna()&(frame.due_date.dt.normalize()<start)
        tr=np.flatnonzero((prior & maturity).to_numpy());va=np.flatnonzero(val.to_numpy())
        if len(tr)<10 or len(va)<10:
            warnings.warn(f'Temporal fold {k+1} too small: train={len(tr)}, val={len(va)}; omitted')
            continue
        assert dates.iloc[tr].max()<dates.iloc[va].min()
        assert frame.due_date.iloc[tr].max().normalize()<dates.iloc[va].min()
        out.append((tr,va))
    return out

def build_folds(frame,task):
    y=frame.target.astype(int).to_numpy() if task=='classification' else frame.Total_Amount.to_numpy()
    groups=frame.customer_id.to_numpy()
    folds={}
    if task=='classification':
        folds['stratified']=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(frame,y))
        folds['grouped']=list(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED).split(frame,y,groups))
    else:
        from sklearn.model_selection import KFold
        folds['stratified']=list(KFold(n_splits=5,shuffle=True,random_state=SEED).split(frame))
        folds['grouped']=list(GroupKFold(n_splits=5).split(frame,y,groups))
    folds['temporal']=build_temporal_folds(frame)
    return folds

cls_folds=build_folds(cls_dev,'classification')
reg_folds=build_folds(reg_dev,'regression')

def save_folds(frame,folds,task):
    records=[];summary=[]
    for protocol,pairs in folds.items():
        for fold_id,(train_idx,val_idx) in enumerate(pairs,1):
            train=frame.iloc[train_idx];val=frame.iloc[val_idx]
            overlap=len(set(train.customer_id)&set(val.customer_id))
            if protocol=='grouped':assert overlap==0
            if protocol=='temporal':assert train.disbursement_date.max()<val.disbursement_date.min()
            for split,sub in [('train',train),('validation',val)]:
                records.extend({'task':task,'protocol':protocol,'fold':fold_id,'partition':split,'row_key':int(key)} for key in sub.row_key)
            row={'task':task,'protocol':protocol,'fold':fold_id,'train_rows':len(train),'val_rows':len(val),'customer_overlap':overlap,
                 'train_max_date':str(train.disbursement_date.max().date()),'val_min_date':str(val.disbursement_date.min().date())}
            if task=='classification': row.update(train_defaults=int(train.target.sum()),val_defaults=int(val.target.sum()),val_default_rate=float(val.target.mean()))
            summary.append(row)
    pd.DataFrame(records).to_csv(SPLITS/(task+'_cv_fold_assignments.csv'),index=False)
    return pd.DataFrame(summary)

fold_report=pd.concat([save_folds(cls_dev,cls_folds,'classification'),save_folds(reg_dev,reg_folds,'regression')],ignore_index=True)
fold_report.to_csv(TABLES/'cv_fold_diagnostics.csv',index=False)
display(fold_report)
if ((fold_report.task=='classification')&((fold_report.get('val_defaults',pd.Series(dtype=float))<3))).any():
    warnings.warn('Some classification folds have very few defaults. Interpret PR-AUC/CI with caution.')

## 4 · EDA and publication-ready development-data figures
These diagnostic figures use development data for modelling insights. Holdout distribution is reported only in coarse aggregate statistics for split auditing. Figures are saved as PNG and PDF.

In [ ]:
def save_figure(name):
    plt.tight_layout()
    plt.savefig(FIGURES/(name+'.png'),dpi=200,bbox_inches='tight')
    plt.savefig(FIGURES/(name+'.pdf'),bbox_inches='tight')
    plt.show()

fig,ax=plt.subplots(figsize=(6.5,4));cnt=cls_dev.target.value_counts().reindex([0,1],fill_value=0)
ax.bar(['Repaid (0)','Defaulted (1)'],cnt.values);ax.set_ylabel('Development loan count');ax.set_title('Default imbalance — development data')
for i,v in enumerate(cnt.values):ax.text(i,v,format(v,','),ha='center',va='bottom')
save_figure('v2_development_class_balance')

fig,ax=plt.subplots(figsize=(7,4));ax.hist(np.log1p(reg_dev.Total_Amount),bins=45);ax.set_title('Historical loan amounts (development)');ax.set_xlabel('log(1 + disbursed amount in KES)');ax.set_ylabel('Frequency');save_figure('v2_development_log_loan_amount')

fig,ax=plt.subplots(figsize=(9,5));feat=feature_sets[FEATURE_SET]
missing=cls_dev[feat].isna().mean().sort_values(ascending=False).head(20).sort_values()
ax.barh(missing.index,100*missing.values);ax.set_xlabel('Missing (%)');ax.set_title('Twenty most missing primary predictors');save_figure('v2_primary_feature_missingness')

fig,ax=plt.subplots(figsize=(9,4));m=cls_dev.groupby(cls_dev.disbursement_date.dt.to_period('M')).agg(loans=('target','size'),defaults=('target','sum'))
m.index=m.index.to_timestamp();ax.plot(m.index,m.loans,label='Loans');ax.set_xlabel('Disbursement month');ax.set_ylabel('Loans');ax.legend();ax.set_title('Development lending volume over time');save_figure('v2_development_loan_volume')

fig,ax=plt.subplots(figsize=(9,4));g=fold_report[fold_report.task=='classification'].copy()
for p,part in g.groupby('protocol'):
    ax.plot(part.fold,part.val_default_rate*100,marker='o',label=p)
ax.set_xlabel('Fold');ax.set_ylabel('Validation default prevalence (%)');ax.set_title('Class prevalence by validation protocol');ax.legend();save_figure('v2_cv_default_prevalence')

## 5 · Model pipelines fitted **inside each fold**
For transparent syllabus coverage, compare **Dummy, Logistic Regression, pruned Decision Tree, Random Forest, HistGradientBoosting** (classification) and **Dummy Mean, OLS, Ridge, Lasso, Random Forest, HistGradientBoosting** (regression). This is a **baseline comparison**, not an exhaustive hyperparameter search.

Categorical `loan_type` is **one-hot encoded inside each fitted pipeline**. Numeric imputation and optional scaling are also fitted inside the fold. Class weighting is implemented as a separate explicit configuration. No global SMOTE, no preprocessing outside CV.

In [ ]:
FEATURES=feature_sets[FEATURE_SET]
CAT=[f for f in FEATURES if f in cat_feats]
NUM=[f for f in FEATURES if f not in CAT]

def make_preprocessor(scale=False):
    numeric=Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True))]+([('scaler',StandardScaler())] if scale else []))
    categorical=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
    return ColumnTransformer([('num',numeric,NUM),('cat',categorical,CAT)],remainder='drop',sparse_threshold=0)

def models_for(task):
    if task=='classification':
        return {
            'dummy_majority':(DummyClassifier(strategy='most_frequent'),False),
            'logistic':(LogisticRegression(max_iter=800,random_state=SEED),True),
            'logistic_balanced':(LogisticRegression(max_iter=800,class_weight='balanced',random_state=SEED),True),
            'decision_tree_pruned':(DecisionTreeClassifier(max_depth=5,min_samples_leaf=25,ccp_alpha=0.0001,random_state=SEED),False),
            'random_forest':(RandomForestClassifier(n_estimators=120,max_depth=12,min_samples_leaf=5,n_jobs=N_JOBS,random_state=SEED),False),
            'hgb':(HistGradientBoostingClassifier(max_iter=90,max_leaf_nodes=15,random_state=SEED),False),
        }
    return {
        'dummy_mean':(DummyRegressor(strategy='mean'),False),
        'ols_linear_regression':(LinearRegression(),True),
        'ridge':(Ridge(alpha=10.0),True),
        'lasso':(Lasso(alpha=5.0,max_iter=3000,random_state=SEED),True),
        'random_forest':(RandomForestRegressor(n_estimators=120,max_depth=12,min_samples_leaf=5,n_jobs=N_JOBS,random_state=SEED),False),
        'hgb':(HistGradientBoostingRegressor(max_iter=90,max_leaf_nodes=15,random_state=SEED),False)
    }

print('Features:',len(FEATURES),'numeric:',len(NUM),'categorical:',CAT)
print('Classification models:',list(models_for('classification')))
print('Regression models:',list(models_for('regression')))

## 6 · Run baseline comparisons and record every fold
The evaluation threshold of 0.5 is reported as a **benchmark only**. It is not a calibrated business decision threshold. Classification PR-AUC is the main discrimination metric. Regression MAE/RMSE are reported in KES; never infer optimal/affordable loan amounts from this historical target.

Fold scores with only one observed class have undefined PR-AUC/ROC-AUC; those values are reported as missing, not invented.

In [ ]:
def score_class(y,p):
    y=np.asarray(y,dtype=int);p=np.clip(np.asarray(p,dtype=float),0,1)
    pred=(p>=0.5).astype(int)
    return {'pr_auc':float(average_precision_score(y,p)) if len(np.unique(y))==2 else np.nan,
            'roc_auc':float(roc_auc_score(y,p)) if len(np.unique(y))==2 else np.nan,
            'precision_05':float(precision_score(y,pred,zero_division=0)),
            'recall_05':float(recall_score(y,pred,zero_division=0)),
            'f1_05':float(f1_score(y,pred,zero_division=0))}
def score_reg(y,p):
    y=np.asarray(y,dtype=float);p=np.asarray(p,dtype=float)
    if not np.isfinite(p).all():raise ValueError('Non-finite regression predictions')
    return {'mae_kes':float(mean_absolute_error(y,p)),
            'rmse_kes':float(np.sqrt(mean_squared_error(y,p))),
            'r2':float(r2_score(y,p)),
            'mae_log':float(mean_absolute_error(np.log1p(y),np.log1p(np.maximum(p,0)))),
            'rmse_log':float(np.sqrt(mean_squared_error(np.log1p(y),np.log1p(np.maximum(p,0)))))}

def run_task(task,frame,folds):
    all_results=[];all_predictions=[];all_errors=[]
    specs=models_for(task)
    ycol='target' if task=='classification' else 'Total_Amount'
    for protocol in CV_PROTOCOLS:
        if protocol not in folds:continue
        for name,(est,scale) in specs.items():
            start=time.time();done=0
            for fold_id,(train_idx,val_idx) in enumerate(folds[protocol],1):
                tr=frame.iloc[train_idx];va=frame.iloc[val_idx]
                if task=='classification' and (tr.target.nunique()<2 or va.target.nunique()<2):
                    all_errors.append({'task':task,'protocol':protocol,'model':name,'fold':fold_id,'error':'Training or validation contains only one class'});continue
                pipe=Pipeline([('preprocess',make_preprocessor(scale)),('model',__import__('sklearn').base.clone(est))])
                try:
                    pipe.fit(tr[FEATURES],tr[ycol])
                    pred=pipe.predict_proba(va[FEATURES])[:,1] if task=='classification' else pipe.predict(va[FEATURES])
                    metrics=score_class(va[ycol],pred) if task=='classification' else score_reg(va[ycol],pred)
                    all_results.append({'task':task,'protocol':protocol,'model':name,'fold':fold_id,'feature_set':FEATURE_SET,
                                        'train_rows':len(tr),'validation_rows':len(va),'runtime_sec':time.time()-start,**metrics})
                    all_predictions.extend({'task':task,'protocol':protocol,'model':name,'fold':fold_id,
                                            'row_key':int(k),'actual':float(y),'prediction':float(p)}
                                           for k,y,p in zip(va.row_key,va[ycol],pred))
                    done+=1
                except Exception as exc:
                    all_errors.append({'task':task,'protocol':protocol,'model':name,'fold':fold_id,'error':repr(exc)})
                    print('ERROR',task,protocol,name,fold_id,repr(exc))
            print(f'{task:14s} {protocol:11s} {name:24s} {done} folds, {time.time()-start:.1f} sec')
    metrics=pd.DataFrame(all_results)
    metrics.to_csv(TABLES/(task+'_fold_metrics.csv'),index=False)
    pd.DataFrame(all_predictions).to_csv(PREDICTIONS/(task+'_validation_predictions.csv'),index=False)
    pd.DataFrame(all_errors,columns=['task','protocol','model','fold','error']).to_csv(TABLES/(task+'_errors.csv'),index=False)
    if metrics.empty:raise RuntimeError('No completed experiments: see errors file')
    metric_cols=['pr_auc','roc_auc','precision_05','recall_05','f1_05'] if task=='classification' else ['mae_kes','rmse_kes','r2','mae_log','rmse_log']
    comparison=metrics.groupby(['task','protocol','model','feature_set'],dropna=False)[metric_cols].agg(['mean','std']).reset_index()
    comparison.columns=['_'.join(x).strip('_') if isinstance(x,tuple) else x for x in comparison.columns.to_flat_index()]
    comparison.to_csv(TABLES/(task+'_comparison.csv'),index=False)
    return metrics,comparison

if RUN_BASELINES:
    print('STARTING CLASSIFICATION...')
    classification_fold_metrics,classification_comparison=run_task('classification',cls_dev,cls_folds)
    print('STARTING REGRESSION...')
    regression_fold_metrics,regression_comparison=run_task('regression',reg_dev,reg_folds)
    display(classification_comparison.sort_values(['protocol','pr_auc_mean'],ascending=[True,False]).head(20))
    display(regression_comparison.sort_values(['protocol','mae_kes_mean']).head(20))
else:
    print('Baseline training disabled: splits/EDA have still been exported.')

## 7 · Comparative research figures
Produce a per-protocol model comparison so that performance under random vs grouped vs temporal validation is never mixed without labels.

In [ ]:
if RUN_BASELINES:
    for task,table,metric,ascending in [('classification',classification_comparison,'pr_auc_mean',False),('regression',regression_comparison,'mae_kes_mean',True)]:
        for protocol in CV_PROTOCOLS:
            sub=table.loc[table.protocol.eq(protocol)].sort_values(metric,ascending=ascending)
            if sub.empty:continue
            fig,ax=plt.subplots(figsize=(9,4))
            ax.bar(sub.model,sub[metric]);ax.set_xticklabels(sub.model,rotation=35,ha='right')
            ax.set_ylabel('Mean PR-AUC' if task=='classification' else 'Mean MAE (KES)')
            ax.set_title(f'V2 {task}: {protocol} CV — {FEATURE_SET}')
            save_figure(f'v2_{task}_{protocol}_comparison')
    registry=pd.concat([classification_fold_metrics,regression_fold_metrics],ignore_index=True,sort=False)
    registry['seed']=SEED;registry['feature_version']='v2';registry['status']='completed'
    registry.to_csv(OUTPUT/'experiment_registry.csv',index=False)
    print('Saved experiment registry:',OUTPUT/'experiment_registry.csv')

## 8 · Final output manifest and essential limitations
This notebook intentionally leaves final holdout predictions untouched. Future notebook steps: threshold sweep and calibrated probabilities (development-only), bootstrapped CIs, fairness and transfer analysis, simple saved scoring service with reason codes, and **one-time** holdout evaluation after selecting a fixed model and threshold.

**Data limitation:** source records do not contain actual application or outcome-confirmation timestamps. Due dates approximate label maturity; this is not proof of when a default became observable. Current `Total_Amount` is historical disbursement, not an economically validated lending recommendation.

In [ ]:
metadata={
    'source':'train.csv only through Feature Engineering V2',
    'feature_set':FEATURE_SET,'features':FEATURES,'categorical':CAT,
    'feature_source_sha256':source_manifest.get('input_sha256'),
    'holdout_date':str(cutoff.date()),
    'seed':SEED,'run_baselines':RUN_BASELINES,'protocols':CV_PROTOCOLS,
    'classification_models':list(models_for('classification')),
    'regression_models':list(models_for('regression')),
    'holdout_policy':'not evaluated in this notebook',
    'data_limitations':['application date approximated by disbursement date','actual outcome confirmation date absent','regression target is historical disbursement'],
}
(OUTPUT/'experiment_setup_manifest.json').write_text(json.dumps(metadata,indent=2,default=str),encoding='utf-8')
print('COMPLETE. Outputs:',OUTPUT.resolve())
print('Next: inspect',TABLES/'classification_comparison.csv',TABLES/'regression_comparison.csv',TABLES/'cv_fold_diagnostics.csv')